# Stage 6-3: Binary 실험 비교

이 노트북은 `experiments/run_all.py`로 실행한 binary task의 MLP와 CNN 학습 결과를 비교한다.

**Binary task 특징**: MNIST 레이블을 `label % 2`로 변환하여 짝수(0) / 홀수(1) 이진 분류를 수행한다.

**실험 설정**
| 모델 | exp_name |
|---|---|
| MLP | `binary_mlp_ep10_lr0.01_bs64` |
| CNN | `binary_cnn_ep10_lr0.001_bs32` |

In [ ]:
# sys.path 설정 — 프로젝트 루트를 모듈 검색 경로에 추가한다.
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tempfile

In [ ]:
from src.data.mnist import MnistDataset, get_task_spec
from src.data.dataloader import DataLoader
from src.models.mlp import MLP
from src.core.evaluator import Evaluator
from src.core.predictor import Predictor
from src.core.visualizer import Visualizer
from src.utils import checkpoints
from src.nn.activations import sigmoid

In [ ]:
# 학습 결과 없으면 run_all.py 실행
TASK = "binary"
MLP_DIR = "../../outputs/binary_mlp_ep10_lr0.01_bs64"
CNN_DIR = "../../outputs/binary_cnn_ep10_lr0.001_bs32"
DATASET_DIR = "/mnt/d/datasets/mnist"

if not os.path.exists(f"{MLP_DIR}/model.npz"):
    print("학습 결과 없음 — run_all.py 실행 중... (수 분 소요)")
    r = subprocess.run(
        [sys.executable, "../../experiments/run_all.py"],
        capture_output=True, text=True,
    )
    print(r.stdout[-1000:])
else:
    print("학습 결과 확인:")
    print(f"  MLP: {MLP_DIR}/model.npz")
    cnn_status = "있음" if os.path.exists(f"{CNN_DIR}/model.npz") else "없음 (CuPy 환경 필요)"
    print(f"  CNN: {cnn_status}")

## 1. Binary 개요

MNIST 레이블을 `label % 2`로 변환하여 짝수(0) / 홀수(1)를 구분하는 이진 분류 task이다.

| 항목 | 내용 |
|---|---|
| 출력 차원 | 1 (이진 출력) |
| loss | binary cross-entropy (sigmoid 내장) |
| metric | binary accuracy (`x >= 0` 트릭) |
| 후처리 | threshold — `sigmoid(logit) >= 0.5` |

**Threshold 트릭**: $\sigma(x) \geq 0.5$는 $x \geq 0$과 동치이다.
sigmoid를 거치지 않고 logit 부호만으로 이진 판정이 가능하다.

In [ ]:
# task_spec 및 데이터 확인
ts = get_task_spec(TASK)
print("task_spec:", ts)

test_ds = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)
print(f"test set: {len(test_ds)} samples")

# 타겟 분포 확인 — 짝수/홀수는 거의 균등하다
targets = test_ds.targets
print(f"타겟 분포: 0(짝수)={int((targets == 0).sum())}, 1(홀수)={int((targets == 1).sum())}")

In [ ]:
# sigmoid threshold 트릭 검증
logits_demo = np.array([-2.0, -0.5, 0.0, 0.5, 2.0], dtype=np.float32)
sigma = sigmoid(logits_demo)
by_sigma = (sigma >= 0.5).astype(int)
by_sign  = (logits_demo >= 0).astype(int)

print("logit:", logits_demo)
print("sigmoid:", sigma.round(3))
print("threshold via sigmoid:", by_sigma)
print("threshold via sign:   ", by_sign)
assert np.array_equal(by_sigma, by_sign), "두 방법이 동일해야 한다."
print("threshold 동치 검증 통과")

## 2. MLP 결과 — 학습 곡선과 예측 grid

In [ ]:
mlp_log = f"{MLP_DIR}/training_log.png"
mlp_pred = f"{MLP_DIR}/predictions.png"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, img_path, title in [
    (axes[0], mlp_log, "MLP: Training Log"),
    (axes[1], mlp_pred, "MLP: Predictions Grid"),
]:
    if os.path.exists(img_path):
        ax.imshow(Image.open(img_path))
    else:
        ax.text(0.5, 0.5, "결과 없음", ha="center", va="center")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Binary MLP (10 epochs, lr=0.01, bs=64)", fontsize=13)
plt.tight_layout()
plt.show()

## 3. MLP checkpoint 로드 후 직접 평가

In [ ]:
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

mlp_model = MLP(task=TASK, seed=42)
if os.path.exists(f"{MLP_DIR}/model.npz"):
    checkpoints.load(mlp_model, f"{MLP_DIR}/model.npz")
    print("MLP checkpoint 로드 완료")

evaluator_mlp = Evaluator(mlp_model, ts)
result_mlp = evaluator_mlp.evaluate(test_loader)
print(f"MLP test: loss={result_mlp['loss']:.4f}, binary_accuracy={result_mlp['metric']:.4f}, samples={result_mlp['num_samples']}")

In [ ]:
# 예측 grid 직접 생성
tmpdir = tempfile.mkdtemp()
images_sample = test_ds.images[:32]
labels_sample = test_ds.labels_raw[:32]

predictor_mlp = Predictor(mlp_model, ts)
pred_mlp = predictor_mlp.predict(images_sample)

vis = Visualizer(output_dir=tmpdir)
grid_path = vis.plot_predictions(
    images=images_sample,
    labels=labels_sample % 2,
    predictions=pred_mlp["predictions"],
    filename="binary_mlp_preds.png",
    n=16,
)

print("예측값 (0=짝수, 1=홀수):", pred_mlp["predictions"][:8])
print("실제 레이블:", labels_sample[:8])
print("짝홀 변환:", labels_sample[:8] % 2)

plt.figure(figsize=(14, 4))
plt.imshow(Image.open(grid_path))
plt.axis("off")
plt.title("Binary MLP 예측 grid (0=짝수, 1=홀수)")
plt.tight_layout()
plt.show()

## 4. MLP vs CNN 비교

In [ ]:
cnn_log = f"{CNN_DIR}/training_log.png"
cnn_pred = f"{CNN_DIR}/predictions.png"

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (img_path, title) in enumerate([
    (mlp_log,  "MLP: Training Log (lr=0.01, bs=64)"),
    (cnn_log,  "CNN: Training Log (lr=0.001, bs=32)"),
    (mlp_pred, "MLP: Predictions Grid"),
    (cnn_pred, "CNN: Predictions Grid"),
]):
    ax = axes[i // 2][i % 2]
    if os.path.exists(img_path):
        ax.imshow(Image.open(img_path))
    else:
        ax.text(0.5, 0.5, "결과 없음", ha="center", va="center", fontsize=14)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Binary: MLP vs CNN 비교", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Multiclass와의 차이 비교

| 항목 | Multiclass | Binary |
|---|---|---|
| 출력 차원 | 10 | 1 |
| 활성화 | softmax (내장) | sigmoid (내장) |
| loss | cross-entropy | binary cross-entropy |
| metric | accuracy (argmax) | binary_accuracy (`x>=0`) |
| 후처리 | argmax | threshold (sigmoid >= 0.5) |
| 난이도 | 높음 (10분류) | 낮음 (2분류) |

Binary task는 2분류이므로 일반적으로 multiclass보다 높은 accuracy를 달성한다.

## 6. 요약

**Binary task**
- 타겟 변환: `label % 2` (짝수=0, 홀수=1)
- sigmoid threshold 예측: $\sigma(x) \geq 0.5 \Leftrightarrow x \geq 0$
- binary_accuracy는 logit 부호만으로 계산 가능하여 sigmoid 불필요
- 2분류이므로 multiclass보다 높은 accuracy 달성이 쉽다

| 모델 | 파라미터 수 | test accuracy (참고) |
|---|---|---|
| MLP | 235,018 | ~0.99 |
| CNN | 824,201 | ~0.99 |